In [4]:
from pathlib import Path
import json
import numpy as np
import numpy as np
import tensorflow as tf
import tensorflow as tf
import xml.etree.ElementTree as ET
from pathlib import Path

BASE_DIR = Path(r"C:\Users\oezde\Desktop\Sampling Notebooks")

MODEL_PATH = BASE_DIR / "best_model.weights.h5"
VOCAB_PATH = BASE_DIR / "vocabulary.json"
MAKAM_TO_ID_PATH = BASE_DIR / "makam_to_id.json"

In [5]:
with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab_list = json.load(f)

with open(MAKAM_TO_ID_PATH, "r", encoding="utf-8") as f:
    makam_to_id = json.load(f)

token_to_idx = {tok: i for i, tok in enumerate(vocab_list)}
idx_to_token = dict(enumerate(vocab_list))

vocab_size = len(vocab_list)
num_makams = len(makam_to_id)

PAD_ID = token_to_idx["<PAD>"]
END_ID = token_to_idx["<END>"]

print("vocab_size:", vocab_size)
print("num_makams:", num_makams)
print("PAD_ID:", PAD_ID)
print("END_ID:", END_ID)

vocab_size: 136
num_makams: 56
PAD_ID: 0
END_ID: 4


In [6]:
INPUT_LEN = 1026

d_model = 128
n_heads = 2
num_layers = 2
dropout = 0.10
d_ff = 128 * 4

max_dist = 256

In [7]:
class RMSNorm(tf.keras.layers.Layer):
    def __init__(self, d_model, eps=1e-8, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.eps = eps

    def build(self, input_shape):
        self.scale = self.add_weight(
            name="scale",
            shape=(self.d_model,),
            initializer="ones",
            trainable=True
        )

    def call(self, x):
        rms = tf.sqrt(tf.reduce_mean(tf.square(x), axis=-1, keepdims=True) + self.eps)
        return x / rms * self.scale

    def get_config(self):
        config = super().get_config()
        config.update({"d_model": self.d_model, "eps": self.eps})
        return config


class GRUFusion(tf.keras.layers.Layer):
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.gru = tf.keras.layers.GRU(d_model, return_sequences=True)

    def call(self, x_old, x_new, training=False):
        return self.gru(tf.concat([x_old, x_new], axis=-1), training=training)

    def get_config(self):
        config = super().get_config()
        config.update({"d_model": self.d_model})
        return config

In [8]:
class RelativePositionBias(tf.keras.layers.Layer):
    def __init__(self, n_heads, max_dist=max_dist, **kwargs):
        super().__init__(**kwargs)
        self.n_heads = n_heads
        self.max_dist = max_dist
        self.emb = tf.keras.layers.Embedding(2 * max_dist + 1, n_heads)

    def call(self, T):
        i = tf.range(T)[:, None]
        j = tf.range(T)[None, :]

        rel = tf.clip_by_value(j - i, -self.max_dist, self.max_dist)
        bias = self.emb(rel + self.max_dist)

        return tf.transpose(bias, [2, 0, 1])

    def get_config(self):
        config = super().get_config()
        config.update({"n_heads": self.n_heads, "max_dist": self.max_dist})
        return config


class RelPosSelfAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, n_heads, dropout=0.1, max_dist=max_dist, **kwargs):
        super().__init__(**kwargs)

        assert d_model % n_heads == 0, "d_model muss durch n_heads teilbar sein"

        self.d_model = d_model
        self.n_heads = n_heads
        self.dropout_rate = dropout
        self.max_dist = max_dist
        self.head_dim = d_model // n_heads

        self.qkv = tf.keras.layers.Dense(3 * d_model, use_bias=False)
        self.out = tf.keras.layers.Dense(d_model)
        self.dropout = tf.keras.layers.Dropout(dropout)
        self.rel_bias = RelativePositionBias(n_heads, max_dist=max_dist)

    def call(self, x, training=False):
        B = tf.shape(x)[0]
        T = tf.shape(x)[1]

        q, k, v = tf.split(self.qkv(x), 3, axis=-1)

        def split_heads(t):
            t = tf.reshape(t, [B, T, self.n_heads, self.head_dim])
            return tf.transpose(t, [0, 2, 1, 3])

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        scores = tf.matmul(q, k, transpose_b=True) * (self.head_dim ** -0.5)
        scores += self.rel_bias(T)[None, :, :, :]

        causal_mask = tf.linalg.band_part(tf.ones((T, T)), -1, 0)
        scores += (1.0 - causal_mask)[None, None, :, :] * -1e9

        attn = self.dropout(tf.nn.softmax(scores, axis=-1), training=training)

        out = tf.matmul(attn, v)
        out = tf.transpose(out, [0, 2, 1, 3])
        out = tf.reshape(out, [B, T, self.d_model])

        return self.out(out)

    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model": self.d_model,
            "n_heads": self.n_heads,
            "dropout": self.dropout_rate,
            "max_dist": self.max_dist
        })
        return config

In [11]:
class TransformerDecoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, max_distance=max_dist, **kwargs):
        super().__init__(**kwargs)

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_ff = d_ff
        self.dropout_rate = dropout
        self.max_distance = max_distance

        self.attn = RelPosSelfAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout,
            max_dist=max_distance
        )

        self.attn_dropout = tf.keras.layers.Dropout(dropout)
        self.gru1 = GRUFusion(d_model)
        self.norm1 = RMSNorm(d_model)

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(d_ff, activation="relu"),
            tf.keras.layers.Dense(d_model),
        ])

        self.ffn_dropout = tf.keras.layers.Dropout(dropout)
        self.gru2 = GRUFusion(d_model)
        self.norm2 = RMSNorm(d_model)

    def call(self, x, training=False):
        attn_out = self.attn(x, training=training)
        attn_out = self.attn_dropout(attn_out, training=training)

        x = self.gru1(x, attn_out, training=training)
        x = self.norm1(x)

        ffn_out = self.ffn(x, training=training)
        ffn_out = self.ffn_dropout(ffn_out, training=training)

        x = self.gru2(x, ffn_out, training=training)
        x = self.norm2(x)

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model": self.d_model,
            "n_heads": self.n_heads,
            "d_ff": self.d_ff,
            "dropout": self.dropout_rate,
            "max_distance": self.max_distance,
        })
        return config

In [12]:
def build_model_for_sampling():
    token_input = tf.keras.Input(
        shape=(INPUT_LEN,),
        dtype=tf.int32,
        name="token_input"
    )

    makam_input = tf.keras.Input(
        shape=(),
        dtype=tf.int32,
        name="makam_input"
    )

    token_emb = tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=d_model,
        name="token_embedding"
    )(token_input)

    makam_emb = tf.keras.layers.Embedding(
        input_dim=num_makams,
        output_dim=d_model,
        name="makam_embedding"
    )(makam_input)

    makam_emb = tf.keras.layers.Reshape(
        (1, d_model),
        name="makam_expand"
    )(makam_emb)

    makam_emb = tf.keras.layers.Lambda(
        lambda m: tf.tile(m, [1, INPUT_LEN, 1]),
        name="makam_repeat"
    )(makam_emb)

    x = tf.keras.layers.Add(
        name="Token_plus_Makam_Embedding"
    )([token_emb, makam_emb])

    x = tf.keras.layers.Dropout(
        dropout,
        name="Dropout-Layer"
    )(x)

    for i in range(num_layers):
        x = TransformerDecoderBlock(
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            dropout=dropout,
            max_distance=max_dist,
            name=f"decoder_block_{i+1}"
        )(x)

    x = tf.keras.layers.Dense(
        d_model,
        name="final_linear_projection"
    )(x)

    x = tf.keras.layers.Dropout(
        dropout,
        name="output_dropout"
    )(x)

    logits = tf.keras.layers.Dense(
        vocab_size,
        name="token_output"
    )(x)

    model = tf.keras.Model(
        inputs={
            "token_input": token_input,
            "makam_input": makam_input
        },
        outputs=logits,
        name="Tarrean_Transformer"
    )

    return model

In [13]:
model = build_model_for_sampling()

model.load_weights(MODEL_PATH)

print("Gewichte geladen.")


Gewichte geladen.


In [14]:
makam_id = makam_to_id["<HICAZ>"]

dummy_tokens = tf.zeros((1, INPUT_LEN), dtype=tf.int32)
dummy_makam = tf.constant([makam_id], dtype=tf.int32)

logits = model({
    "token_input": dummy_tokens,
    "makam_input": dummy_makam
}, training=False)

print(logits.shape)

(1, 1026, 136)


In [15]:
seed_tokens = [
    "<START>",
    "REL_KOMA:22", "DUR:1", "<SEP>",
    "REL_KOMA:17",  "DUR:1", "<SEP>",
    "REL_KOMA:12",  "DUR:1", "<SEP>",

]

In [16]:
def sample_next_token(logits, temperature=0.8, top_k=8):
    logits = logits.astype(np.float64) / temperature

    if top_k:
        top_idx = np.argpartition(logits, -top_k)[-top_k:]
        mask = np.full_like(logits, -np.inf)
        mask[top_idx] = logits[top_idx]
        logits = mask

    probs = tf.nn.softmax(logits).numpy()
    return int(np.random.choice(len(probs), p=probs))


def generate_free(
    makam,
    seed_tokens,
    max_new_tokens=20,
    min_tokens=45,
    temperature=0.8,
    top_k=6,
):
    makam_id = makam_to_id[f"<{makam.upper()}>"]
    generated = [token_to_idx[t] for t in seed_tokens]

    for _ in range(max_new_tokens):
        context = generated[-INPUT_LEN:]
        context = [PAD_ID] * max(0, INPUT_LEN - len(context)) + context

        logits = model(
            {
                "token_input": tf.constant([context], dtype=tf.int32),
                "makam_input": tf.constant([makam_id], dtype=tf.int32),
            },
            training=False,
        )

        next_id = sample_next_token(
            logits[0, -1, :].numpy(),
            temperature=temperature,
            top_k=top_k,
        )

        generated.append(next_id)

        if next_id == END_ID and len(generated) >= min_tokens:
            break

    return [idx_to_token[i] for i in generated]

In [17]:
tokens_top = generate_free("HICAZ", seed_tokens)

print(" ".join(tokens_top))

<START> REL_KOMA:22 DUR:1 <SEP> REL_KOMA:17 DUR:1 <SEP> REL_KOMA:12 DUR:1 <SEP> <REST> DUR:1 <SEP> REL_KOMA:-9 DUR:1 <SEP> REL_KOMA:0 DUR:1 <SEP> REL_KOMA:0 DUR:1 <SEP> REL_KOMA:-9 DUR:1 <SEP> <REST> DUR:0.5 <SEP> REL_KOMA:0 DUR:1


In [18]:
def tokens_to_events(tokens):
    events = []
    ev = {}

    for tok in tokens:
        if tok in ["<START>", "<PAD>", "<UNK>", "<UNK_EXTRA>"]:
            continue
        if tok == "<END>":
            break

        if tok.startswith("REL_KOMA:"):
            ev = {
                "type": "note",
                "rel_koma": float(tok.split(":")[1])
            }

        elif tok == "<REST>":
            ev = {
                "type": "rest"
            }

        elif tok.startswith("DUR:") and "type" in ev:
            ev["dur"] = float(tok.split(":")[1])

        elif tok == "<SEP>":
            if "type" in ev and "dur" in ev:
                events.append(ev)
            ev = {}

    return events

In [19]:
events = tokens_to_events(tokens_top)
for ev in events[:20]:
    print(ev)

{'type': 'note', 'rel_koma': 22.0, 'dur': 1.0}
{'type': 'note', 'rel_koma': 17.0, 'dur': 1.0}
{'type': 'note', 'rel_koma': 12.0, 'dur': 1.0}
{'type': 'rest', 'dur': 1.0}
{'type': 'note', 'rel_koma': -9.0, 'dur': 1.0}
{'type': 'note', 'rel_koma': 0.0, 'dur': 1.0}
{'type': 'note', 'rel_koma': 0.0, 'dur': 1.0}
{'type': 'note', 'rel_koma': -9.0, 'dur': 1.0}
{'type': 'rest', 'dur': 0.5}


In [39]:
# Eingabe der Tonika

tonika_note = "A"
tonika_oktave = 4

KOMA_PER_OCTAVE = 53


# Step -> Perde
STEP_TO_PERDE = {
    "G": "RAST",
    "A": "DUGAH",
    "B": "BUSELIK",
    "C": "CARGAH",
    "D": "NEVA",
    "E": "HUSEYNI",
    "F": "ACEM",
}

PERDE_BASE_KOMA = {
    "RAST": 0,
    "DUGAH": 9,
    "BUSELIK": 18,
    "CARGAH": 22,
    "NEVA": 31,
    "HUSEYNI": 40,
    "ACEM": 44,
}



def tokens_to_events(tokens):

    events = []
    current_event = {}

    for tok in tokens:

        if tok in ["<START>", "<PAD>", "<UNK>", "<UNK_EXTRA>"]:
            continue

        if tok == "<END>":
            break

        if tok.startswith("REL_KOMA:"):
            current_event["type"] = "note"
            current_event["rel_koma"] = float(tok.split(":")[1])

        elif tok == "<REST>":
            current_event["type"] = "rest"

        elif tok.startswith("DUR:"):
            current_event["dur"] = float(tok.split(":")[1])

        elif tok == "<SEP>":
            if "type" in current_event and "dur" in current_event:
                events.append(current_event)

            current_event = {}

    return events


# -----------------------------------------
# Tonika -> absolute Koma
# -----------------------------------------

def tonic_to_absolute_koma():

    perde = STEP_TO_PERDE[tonika_note]
    tonic_local = PERDE_BASE_KOMA[perde]

    return tonika_oktave * KOMA_PER_OCTAVE + tonic_local


# -----------------------------------------
# REL_KOMA -> absolute
# -----------------------------------------

def rel_to_absolute(rel_koma):

    return tonic_to_absolute_koma() + rel_koma


# -----------------------------------------
# absolute -> split
# -----------------------------------------

def split_koma(absolute_koma):

    octave = int(absolute_koma // KOMA_PER_OCTAVE)
    koma_in_octave = absolute_koma % KOMA_PER_OCTAVE

    return koma_in_octave, octave


# -----------------------------------------
# Events erweitern
# -----------------------------------------

def add_absolute_koma_to_events(events):

    new_events = []

    for ev in events:

        if ev["type"] == "note":

            absolute_koma = rel_to_absolute(ev["rel_koma"])
            koma_in_octave, octave = split_koma(absolute_koma)

            new_events.append({
                "type": "note",
                "rel_koma": ev["rel_koma"],
                "absolute_koma": absolute_koma,
                "koma_in_octave": koma_in_octave,
                "octave": octave,
                "dur": ev["dur"],
            })

        elif ev["type"] == "rest":

            new_events.append({
                "type": "rest",
                "dur": ev["dur"],
            })

    return new_events


# -----------------------------------------
# Benutzung
# -----------------------------------------

events = tokens_to_events(tokens_top)

events_abs = add_absolute_koma_to_events(events)

for ev in events_abs[:40]:
    print(ev)

{'type': 'note', 'rel_koma': 22.0, 'absolute_koma': 243.0, 'koma_in_octave': 31.0, 'octave': 4, 'dur': 1.0}
{'type': 'note', 'rel_koma': 17.0, 'absolute_koma': 238.0, 'koma_in_octave': 26.0, 'octave': 4, 'dur': 1.0}
{'type': 'note', 'rel_koma': 12.0, 'absolute_koma': 233.0, 'koma_in_octave': 21.0, 'octave': 4, 'dur': 1.0}
{'type': 'note', 'rel_koma': 22.0, 'absolute_koma': 243.0, 'koma_in_octave': 31.0, 'octave': 4, 'dur': 1.0}
{'type': 'note', 'rel_koma': 17.0, 'absolute_koma': 238.0, 'koma_in_octave': 26.0, 'octave': 4, 'dur': 1.0}
{'type': 'rest', 'dur': 0.5}
{'type': 'note', 'rel_koma': 5.0, 'absolute_koma': 226.0, 'koma_in_octave': 14.0, 'octave': 4, 'dur': 1.0}
{'type': 'note', 'rel_koma': 5.0, 'absolute_koma': 226.0, 'koma_in_octave': 14.0, 'octave': 4, 'dur': 0.5}
{'type': 'note', 'rel_koma': 0.0, 'absolute_koma': 221.0, 'koma_in_octave': 9.0, 'octave': 4, 'dur': 0.5}
{'type': 'note', 'rel_koma': 0.0, 'absolute_koma': 221.0, 'koma_in_octave': 9.0, 'octave': 4, 'dur': 0.5}
{'typ

In [40]:
# -----------------------------------------
# Natürliche Noten
# -----------------------------------------

KOMA_TO_NOTE = {
    0.0: "G",
    9.0: "A",
    18.0: "B",
    22.0: "C",
    31.0: "D",
    40.0: "E",
    44.0: "F",
}


# -----------------------------------------
# Rundung
# -----------------------------------------

def q(x, nd=1):
    if x is None:
        return "NONE"

    x = round(float(x), nd)

    return int(x) if x.is_integer() else x


# -----------------------------------------
# Differenz -> accidental
# -----------------------------------------

ACCIDENTAL_MAP = {
    -8: "three-quarters-flat",
    -5: "slash-quarter-flat",
    -4: "slash-flat",
    -1: "quarter-flat",

     0: "natural",

     1: "quarter-sharp",
     4: "slash-sharp",
     5: "slash-quarter-sharp",
     8: "three-quarters-sharp",
}


def get_accidental_from_difference(diff):
    return ACCIDENTAL_MAP.get(q(diff, 1), None)


# -----------------------------------------
# nächste natürliche Note suchen
# -----------------------------------------

def get_koma_difference(koma_in_octave):

    kleinste = None
    note_name = None
    nearest = None

    for koma_wert, note in KOMA_TO_NOTE.items():

        diff = round(koma_in_octave - koma_wert, 3)

        if kleinste is None or abs(diff) < abs(kleinste):
            kleinste = diff
            note_name = note
            nearest = koma_wert

    return kleinste, note_name, nearest


# -----------------------------------------
# Events ergänzen
# -----------------------------------------

def add_note_name(events_abs):

    neue_events = []

    for ev in events_abs:

        ev = ev.copy()

        # REST
        if ev["type"] == "rest":

            ev["note_name"] = None
            ev["koma_difference"] = None
            ev["nearest_note"] = None
            ev["nearest_koma"] = None
            ev["accidental_name"] = None
            ev["xml_octave"] = None

            neue_events.append(ev)
            continue

        wert = round(ev["koma_in_octave"], 3)

        diff, note_name, nearest_koma = get_koma_difference(wert)

        # MusicXML Oktave korrigieren:
        # C D E F gehören über A/B
        xml_octave = ev["octave"]

        if note_name in ["C", "D", "E", "F"]:
            xml_octave += 1

        ev["note_name"] = note_name
        ev["koma_difference"] = diff
        ev["nearest_note"] = note_name
        ev["nearest_koma"] = nearest_koma
        ev["accidental_name"] = get_accidental_from_difference(diff)
        ev["xml_octave"] = xml_octave

        neue_events.append(ev)

    return neue_events


# -----------------------------------------
# Benutzung
# -----------------------------------------

events_notes = add_note_name(events_abs)

for ev in events_notes[:150]:
    print(ev)

{'type': 'note', 'rel_koma': 22.0, 'absolute_koma': 243.0, 'koma_in_octave': 31.0, 'octave': 4, 'dur': 1.0, 'note_name': 'D', 'koma_difference': 0.0, 'nearest_note': 'D', 'nearest_koma': 31.0, 'accidental_name': 'natural', 'xml_octave': 5}
{'type': 'note', 'rel_koma': 17.0, 'absolute_koma': 238.0, 'koma_in_octave': 26.0, 'octave': 4, 'dur': 1.0, 'note_name': 'C', 'koma_difference': 4.0, 'nearest_note': 'C', 'nearest_koma': 22.0, 'accidental_name': 'slash-sharp', 'xml_octave': 5}
{'type': 'note', 'rel_koma': 12.0, 'absolute_koma': 233.0, 'koma_in_octave': 21.0, 'octave': 4, 'dur': 1.0, 'note_name': 'C', 'koma_difference': -1.0, 'nearest_note': 'C', 'nearest_koma': 22.0, 'accidental_name': 'quarter-flat', 'xml_octave': 5}
{'type': 'note', 'rel_koma': 22.0, 'absolute_koma': 243.0, 'koma_in_octave': 31.0, 'octave': 4, 'dur': 1.0, 'note_name': 'D', 'koma_difference': 0.0, 'nearest_note': 'D', 'nearest_koma': 31.0, 'accidental_name': 'natural', 'xml_octave': 5}
{'type': 'note', 'rel_koma': 1

In [41]:


def dur_to_xml_duration(dur, divisions=10):
    return int(round(float(dur) * divisions))


def dur_to_xml_type(dur):

    dur = float(dur)

    if dur >= 4:
        return "whole"
    elif dur >= 2:
        return "half"
    elif dur >= 1:
        return "quarter"
    elif dur >= 0.5:
        return "eighth"
    elif dur >= 0.25:
        return "16th"
    else:
        return "32nd"


def koma_diff_to_alter(diff):

    mapping = {
        -9: -1.0,
        -6: -0.75,
        -4.5: -0.5,
        -4: -0.25,
        -2: -0.125,

         0: 0,

         2: 0.125,
         4.5: 0.5,
         5: 0.625,
         6: 0.75,
         8: 0.875,
         9: 1.0,
    }

    return mapping.get(diff, 0)


def events_to_musicxml(events_notes, output_path):

    score = ET.Element("score-partwise", version="3.1")

    part_list = ET.SubElement(score, "part-list")
    score_part = ET.SubElement(part_list, "score-part", id="P1")

    ET.SubElement(score_part, "part-name").text = "Generated Makam"

    part = ET.SubElement(score, "part", id="P1")
    measure = ET.SubElement(part, "measure", number="1")

    attributes = ET.SubElement(measure, "attributes")
    ET.SubElement(attributes, "divisions").text = "10"

    for ev in events_notes:

        note_el = ET.SubElement(measure, "note")

        if ev["type"] == "rest":

            ET.SubElement(note_el, "rest")

        else:

            pitch = ET.SubElement(note_el, "pitch")

            ET.SubElement(pitch, "step").text = ev["note_name"]

            alter_value = koma_diff_to_alter(ev["koma_difference"])

            if alter_value != 0:
                ET.SubElement(pitch, "alter").text = str(alter_value)

            ET.SubElement(
                pitch,
                "octave"
            ).text = str(ev["xml_octave"])
                
            if ev["accidental_name"] not in [None, "natural"]:
                ET.SubElement(note_el, "accidental").text = ev["accidental_name"]

        ET.SubElement(
            note_el,
            "duration"
        ).text = str(dur_to_xml_duration(ev["dur"]))

        ET.SubElement(
            note_el,
            "type"
        ).text = dur_to_xml_type(ev["dur"])

    ET.indent(score, space="  ")

    tree = ET.ElementTree(score)
    tree.write(output_path, encoding="utf-8", xml_declaration=True)

    print(output_path)

In [42]:
events = tokens_to_events(tokens_top)

events_abs = add_absolute_koma_to_events(events)

events_notes = add_note_name(events_abs)

for ev in events_notes[:80]:
    print(ev)

{'type': 'note', 'rel_koma': 22.0, 'absolute_koma': 243.0, 'koma_in_octave': 31.0, 'octave': 4, 'dur': 1.0, 'note_name': 'D', 'koma_difference': 0.0, 'nearest_note': 'D', 'nearest_koma': 31.0, 'accidental_name': 'natural', 'xml_octave': 5}
{'type': 'note', 'rel_koma': 17.0, 'absolute_koma': 238.0, 'koma_in_octave': 26.0, 'octave': 4, 'dur': 1.0, 'note_name': 'C', 'koma_difference': 4.0, 'nearest_note': 'C', 'nearest_koma': 22.0, 'accidental_name': 'slash-sharp', 'xml_octave': 5}
{'type': 'note', 'rel_koma': 12.0, 'absolute_koma': 233.0, 'koma_in_octave': 21.0, 'octave': 4, 'dur': 1.0, 'note_name': 'C', 'koma_difference': -1.0, 'nearest_note': 'C', 'nearest_koma': 22.0, 'accidental_name': 'quarter-flat', 'xml_octave': 5}
{'type': 'note', 'rel_koma': 22.0, 'absolute_koma': 243.0, 'koma_in_octave': 31.0, 'octave': 4, 'dur': 1.0, 'note_name': 'D', 'koma_difference': 0.0, 'nearest_note': 'D', 'nearest_koma': 31.0, 'accidental_name': 'natural', 'xml_octave': 5}
{'type': 'note', 'rel_koma': 1

In [43]:
import random

# einmal pro Stück erzeugen
piece_id = random.randint(1000, 9999)

print("Aktuelle Stück-ID:", piece_id)

import os

BASE_DIR = r"C:\Users\oezde\Desktop\Sampling Notebooks"

output_path = os.path.join(
    BASE_DIR,
    f"hicaz_new_{piece_id}.musicxml"
)

events_to_musicxml(
    events_notes,
    output_path
)

print("Gespeichert:", output_path)

Aktuelle Stück-ID: 4195
C:\Users\oezde\Desktop\Sampling Notebooks\hicaz_new_4195.musicxml
Gespeichert: C:\Users\oezde\Desktop\Sampling Notebooks\hicaz_new_4195.musicxml
